# End-to-end ML pipeline (ClickHouse)

Sketch: load `user_features`, train/evaluate models, optional time-series forecast, write `user_predictions`.

Prereqs: `user_features` populated; `pip install -r ../requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

SRC = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"
SRC = SRC.resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from db.clickhouse_client import ClickHouseConfig, get_client, insert_rows

cfg = ClickHouseConfig(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username=os.environ.get("CLICKHOUSE_USER", "default"),
    password=os.environ.get("CLICKHOUSE_PASSWORD", ""),
    database="engagement",
)
client = get_client(cfg)
snapshot = "2025-01-15"  # change to your snapshot_date

df = client.query_df(f"""
SELECT *
FROM user_features
WHERE snapshot_date = toDate('{snapshot}')
""")
df.head()

## Train / evaluate (extend here)

- Classification: `high_engagement` vs feature columns.
- Regression: predict `clicks_next_day` or total clicks.
- Insert predictions into `user_predictions` with `insert_rows` or `client.insert`.